# Solution E1 · The neighbors feature that contains the label of the point itself

**Type**: Find the bug

## Objective

Find out why a random forest to which "the label of the nearest neighbor" was added as a feature relies blindly on that column in training and does worse in test: for the **training** points the nearest neighbor is the point itself.
Fix it by excluding itself and measure how much the defect weighs.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors

X, y = make_moons(1500, noise=0.4, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=0)
nn = NearestNeighbors(n_neighbors=1).fit(X_tr)
rf = lambda: RandomForestClassifier(200, random_state=0, n_jobs=-1)

## Commented solution

In [ ]:
f_tr = y_tr[nn.kneighbors(X_tr)[1][:, 0]]            # label of the nearest neighbor, for every training point
f_te = y_tr[nn.kneighbors(X_te)[1][:, 0]]            # and for every test point
A_tr, A_te = np.c_[X_tr, f_tr], np.c_[X_te, f_te]
bug_model = rf().fit(A_tr, y_tr)
bug_test_acc = bug_model.score(A_te, y_te)
bug_imp = bug_model.feature_importances_[-1]
print(f"test accuracy: {bug_test_acc:.3f} | importance of the new column: {bug_imp:.2f}")

**Where the bug is.** `nn.kneighbors(X_tr)` searches the neighbors of every training point **inside the training set**: the nearest neighbor of a point is the point itself, at distance 0. The column `f_tr` is therefore identical to `y_tr`. The forest learns
"copy the new column" (importance 0.67). On the test set the same column is the label of a **different** point and is right only 80% of the time: the model relies on information that it will not have in production.

In [ ]:
train_indices = nn.kneighbors(X_tr)[1][:, 0]
self_share = float((train_indices == np.arange(len(X_tr))).mean())
train_feature_acc = float((f_tr == y_tr).mean())
test_feature_acc = float((f_te == y_te).mean())
print(f"points whose neighbor is itself: {self_share:.0%} | accuracy of the feature: training {train_feature_acc:.2f}, test {test_feature_acc:.2f}")

In [ ]:
def neighbor_feature(nn, Xq, y_ref, exclude_self):
    if exclude_self:
        indices = NearestNeighbors(n_neighbors=2).fit(Xq).kneighbors(Xq)[1][:, 1]      # Xq is the training set: column 0 = itself, column 1 = first real neighbor
    else:
        indices = nn.kneighbors(Xq)[1][:, 0]
    return y_ref[indices]

In [ ]:
A_tr_ok = np.c_[X_tr, neighbor_feature(nn, X_tr, y_tr, True)]
A_te_ok = np.c_[X_te, neighbor_feature(nn, X_te, y_tr, False)]
ok_model = rf().fit(A_tr_ok, y_tr)
ok_test_acc = ok_model.score(A_te_ok, y_te)
ok_imp = ok_model.feature_importances_[-1]
X_only_acc = rf().fit(X_tr, y_tr).score(X_te, y_te)
print(f"test accuracy: bug {bug_test_acc:.3f} -> fixed {ok_test_acc:.3f} (coordinates only: {X_only_acc:.3f}) | importance of the column: {bug_imp:.2f} -> {ok_imp:.2f}")

## Check

In [ ]:
with verify():
    check(bug_test_acc < 0.82, "the colleague's model is below 0.82 on the test set", "don't modify the colleague's code")
    check(self_share == 1.0 and train_feature_acc == 1.0, "for all the training points the 'nearest neighbor' is itself: the feature is the label", "compare the returned indices with np.arange(len(X_tr))")
    check(test_feature_acc < 0.85, "on the test set the same feature is much less accurate (below 0.85)", "the feature only holds for the label of the neighbor in the training set")
    check(np.array_equal(neighbor_feature(nn, X_tr, y_tr, True), y_tr[NearestNeighbors(n_neighbors=2).fit(X_tr).kneighbors(X_tr)[1][:, 1]]) and np.array_equal(neighbor_feature(nn, X_te, y_tr, False), f_te), "neighbor_feature skips itself only when needed", "for the training set query two neighbors and take the second")
    check(ok_test_acc > bug_test_acc + 0.02, "with the correct feature the test accuracy goes up by over 0.02", "exclude_self=True for the training set, False for the test set")
    check(bug_imp > 2.5 * ok_imp, "the colleague's model gives the new column over 2.5 times the importance of the correct one", "feature_importances_[-1]")
    check(abs(ok_test_acc - X_only_acc) < 0.03, "the neighbors feature, done right, adds nothing to the coordinates (gap below 0.03)", "the forest already had the two coordinates")
done()

## What you learned

- **For the training points the nearest neighbor is the point itself.** Every feature built from the target of the neighbors (label of the neighbor, local mean of the target, fraction of positives) computed **on the same training set** contains the answer. The downstream
  model exploits it 100% and in production, where the point is not in the training set, the feature is much less informative.
- The effect is measurable: the colleague's feature is worth 1.00 in training and 0.8 in test, the forest assigns it 67% of the importance (against 19% in the correct model) and the test accuracy drops by four points.
- The fix is to exclude the point itself (`n_neighbors + 1` and skip the first column) or, better, to build the feature **out of sample** with cross-validation (`cross_val_predict` of a KNN): it is the rule of stacking and of target encoding.
- With the feature done right nothing is gained: the forest already sees the coordinates and the neighbors feature adds no information. Before building a complicated feature, ask yourself whether the downstream model already has the same information.

## Common pitfall

Building "proximity features" with `kneighbors(X_train)` and using them in training. Every time you evaluate a KNN or a derivative of it on the training set, the nearest neighbor is you.

## Interview question

> *Click on a question to show its answer.*

<details><summary>How do you do stacking (or target encoding) without leakage?</summary>

The features built from the target must be computed **out of sample**: for every row of the training set I use a model or statistic trained on the other rows (K-fold: `cross_val_predict`, target encoding with folds). For the test set I use the model trained on the whole training set.
That way the distribution of the feature in training and in test coincides and the next model learns to trust it by the right amount.

</details>